#  Обзор бизнес-показателей маркетплейса

Проект: Маркетплейс. Бизнес просит оценить текущее состояние продукта. Выручка бизнеса — 5% комиссии от стоимости любой продажи. Определить, соответствует ли продукт базовым ожиданиям, виден ли рост, есть ли тревожные сигналы. Посмотреть вглубь данных, сформировать обоснованные гипотезы. Сделать выводы, чтобы заказчик мог быстро ознакомиться с основными результатами.

Задачи: 
- Обзор метрик.
- Оценка монетизации и юнит-экономики.
- Поиск инсайтов и точек роста.
- A/B-тестирование.


### Данные

Работа с четерьмя датасетами:  

1. Датасет `/datasets/pa_marketplace_orders.csv` содержит данные о заказах пользователей.
2. Датасет `/datasets/pa_marketplace_campaign_costs.csv` содержит данные о каналах привлечения и маркетинговых тратах.
3. Датасет `/datasets/pa_marketplace_users.csv` содержит данные о пользователях.
4. Датасет `/datasets/pa_marketplace_events.csv` содержит данные о событиях.

Поля датасета `Orders`:
- `order_id` — уникальный идентификатор заказа;
- `user_id` — идентификатор пользователя, который сделал заказ;
- `order_date` — дата и время оформления заказа;
- `product_name` — наименование товара;
- `quantity` — количество единиц товара в заказе;
- `unit_price` — цена за одну единицу товара;
- `total_price` — итоговая сумма заказа;
- `category_name` — наименование категории товара;
- `order_week` — неделя заказа; 
- `order_month` — месяц заказа.

Поля датасета `Campaign_costs`:
- `acq_channel` — канал привлечения;
- `spend_month` — месяц, в который был потрачен бюджет (отражает период, в котором были привлечены пользователи);
- `budget` — маркетинговый бюджет (в денежном выражении), потраченный на данный канал в указанном месяце.

Поля датасета `Users`:
- `user_id` — уникальный идентификатор пользователя;
- `registration_date` — дата регистрации пользователя;
- `age` — возраст пользователя;
- `gender` — пол;
- `region` — регион;
- `acq_channel` — канал привлечения;
- `buyer_segment` — сегмент покупателя;
- `cohort_week` — неделя заказа;
- `cohort_month` — месяц заказа.

Поля датасета `Events`:
- `event_id` — уникальный идентификатор события;
- `user_id` — идентификатор пользователя;
- `event_date` — дата события;
- `event_type` — тип события;
- `os` — операционная система;
- `device` — тип устройства;
- `product_name` — наименование товара, к которому относится событие (если применимо);
- `order_week` — неделя заказа; 
- `order_month` — месяц заказа.

Описание типов событий:
- `page_view` — Открытие любой страницы сайта или приложения пользователем;
- `product_view` — Просмотр страницы конкретного товара;
- `product_click` — Клик по товару (например, из списка товаров или на баннере);
- `add_to_cart` — Добавление товара в корзину;
- `remove_from_cart` — Удаление товара из корзины;
- `search` — Выполнение поиска по сайту или приложению;
- `filter_apply` — Применение фильтра (например, по цене, бренду, категории);
- `checkout_start` — Начало оформления заказа (переход к оформлению корзины);
- `checkout_complete` — Завершение оформления заказа (не гарантирует покупку, покупка зависит от факта оплаты);
- `user_login` — Вход пользователя в личный кабинет или аккаунт;
- `user_logout` — Выход пользователя из аккаунта;
- `wishlist_add` — Добавление товара в список желаемого (wishlist);
- `category_view` — Просмотр страницы категории товаров;
- `banner_click` — Клик по рекламному баннеру на сайте или в приложении;
- `promo_view` — Просмотр промо-акции или специального предложения.


### План проекта
1. Загрузка данных и их предобработка
2. Обзор ключевых метрик
3. Оценка метрик монетизации и юнит-экономики
5. Поиск инсайтов, точек роста и сегментация
6. Подготовка эксперимента и подведение его результатов
7. Выводы

## Загрузка данных и их предобработка

Загрузим данные из датасетов и получим основную информацию о них.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.dates as mdates
from matplotlib import colors as mcolors
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.proportion import proportions_ztest
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import norm, t
from scipy.stats import ttest_ind

In [ ]:
df_orders = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_orders.csv', parse_dates = ['order_date','order_week','order_month'])
df_costs = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_campaign_costs.csv')
df_users = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_users.csv', parse_dates = ['registration_date','cohort_week','cohort_month'])
df_events = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_events.csv', parse_dates = ['event_date','event_week','event_month'])

Начнём с таблицы `df_orders`, содержащей данные о заказах.

In [ ]:
df_orders.head(10)

In [ ]:
df_orders.info()

In [ ]:
df_orders.duplicated().sum()

In [ ]:
df_orders_dubl = df_orders[df_orders.duplicated(subset = ['order_id', 'user_id', 'order_date'], keep = False)]
df_orders_dubl

In [ ]:
df_orders['order_date'].min()

In [ ]:
df_orders['order_date'].max()

In [ ]:
mask_time_err_orders = df_orders['order_date'] < df_orders['order_date']
df_orders[mask_time_err_orders]

In [ ]:
unique_user_id_orders = df_orders['user_id'].nunique()
unique_user_id_orders

In [ ]:
unique_order_id_orders = df_orders['order_id'].nunique()
unique_order_id_orders

In [ ]:
duplicate_user_id_orders = df_orders['user_id'].value_counts()
duplicate_user_id_orders = duplicate_user_id_orders[duplicate_user_id_orders > 1]
duplicate_user_id_orders

In [ ]:
uuid_orders = 11525
uuid_act_orders = df_orders[df_orders['user_id'] == uuid_orders]
uuid_act_orders

In [ ]:
df_orders['product_name'].value_counts()

In [ ]:
df_orders['category_name'].value_counts()

In [ ]:
df_orders['quantity'].value_counts()

In [ ]:
df_orders['unit_price'].describe()

In [ ]:
look_orders_unit_price = df_orders.boxplot(
    column = 'unit_price',
    vert = False,
    showfliers = True
)

fig = plt.gcf()
fig.set_size_inches(15, 4)

look_orders_unit_price.set_title('Цена еденицы товара', fontsize = 14)
look_orders_unit_price.set_xlabel('Цена')

plt.show()

In [ ]:
# def iqr_mask(group):
#     q1 = group.quantile(0.25)
#     q3 = group.quantile(0.75)
#     iqr = q3 - q1
#     lower_bound = q1 - 3 * iqr
#     upper_bound = q3 + 3 * iqr
#     in_range_mask = (group >= lower_bound) & (group <= upper_bound)
#     nan_mask = group.isna()
#     return in_range_mask | nan_mask
# 
# df_mask = iqr_mask(df_orders['unit_price'])
# df_orders = df_orders[df_mask]
# df_orders = df_orders.reset_index(drop = True)

In [ ]:
big_unit_price = df_orders[df_orders['unit_price'] > 60000]
big_unit_price

In [ ]:
df_orders['total_price'].describe()

In [ ]:
look_orders_total_price = df_orders.boxplot(
    column = 'total_price',
    vert = False,
    showfliers = True
)

fig = plt.gcf()
fig.set_size_inches(15, 4)

look_orders_total_price.set_title('Цена заказа', fontsize = 14)
look_orders_total_price.set_xlabel('Цена')

plt.show()

In [ ]:
big_total_price = df_orders[df_orders['total_price'] > 100000]
big_total_price

In [ ]:
df_orders['order_week'].value_counts()

In [ ]:
df_orders['order_month'].value_counts()

Таблица `df_costs`, содержит данные о маркетинговых затратах и каналах привлечения.

In [ ]:
df_costs = df_costs.drop(columns = ['Unnamed: 0'])

In [ ]:
df_costs.head()

In [ ]:
df_costs.info()

In [ ]:
df_costs.duplicated().sum()

In [ ]:
df_costs['acq_channel'].value_counts()

In [ ]:
df_costs['spend_month'].value_counts()

In [ ]:
df_costs['budget'].describe()

In [ ]:
look_costs_budget = df_costs.boxplot(
    column = 'budget',
    vert = False,
    showfliers = True
)

fig = plt.gcf()
fig.set_size_inches(15, 4)

look_costs_budget.set_title('Цена заказа', fontsize = 14)
look_costs_budget.set_xlabel('Цена')

plt.show()

Таблица `df_users`, содержит данные о пользователях.

In [ ]:
df_users.head()

In [ ]:
df_users.info()

In [ ]:
df_users.duplicated().sum()

In [ ]:
df_users['registration_date'].min()

In [ ]:
df_users['registration_date'].max()

In [ ]:
unique_user_id_users = df_users['user_id'].nunique()
unique_user_id_users

In [ ]:
df_users['age'].value_counts().sort_index(ascending = True)

In [ ]:
df_users['gender'].value_counts()

In [ ]:
df_users['region'].value_counts()

In [ ]:
df_users['acq_channel'].value_counts()

In [ ]:
df_users['buyer_segment'].value_counts()

Таблица `df_events`, содержит данные о событиях.

In [ ]:
df_events.head()

In [ ]:
df_events.info()

In [ ]:
df_events.duplicated().sum()

In [ ]:
df_events_dubl = df_events[df_events.duplicated(subset = ['event_id', 'user_id', 'event_date'], keep = False)]
df_events_dubl

In [ ]:
df_events['event_date'].min()

In [ ]:
df_events['event_date'].max()

In [ ]:
unique_user_id_events = df_events['user_id'].nunique()
unique_user_id_events

In [ ]:
unique_event_id_events = df_events['event_id'].nunique()
unique_event_id_events

In [ ]:
steps_df_events = df_events[df_events['user_id'] == 100]
steps_df_events.head(20)

In [ ]:
df_events['event_type'].value_counts()

In [ ]:
df_events['os'] = df_events['os'].str.lower()

In [ ]:
df_events['os'].value_counts()

In [ ]:
df_events['device'].value_counts()

In [ ]:
df_events['product_name'].value_counts()

Данные качественные, дубликатов нет, пропуски не критичные (поле `product_name` в таблице событий). Типы данных соответствуют содержанию.

## Обзор ключевых метрик

Выполним верхнеуровневую оценку состояния продукта с помощью собранных витрин. Бизнес просит оценить текущее состояние — без глубокой сегментации и оценки сложных метрик. Выручка бизнеса — 5% комиссии от стоимости любой продажи. 

Посмотрим на общую динамику ключевых метрик.

In [ ]:
dau = (
    df_events
    .sort_values('event_date')
    .groupby('event_date')['user_id']
    .nunique()
    .reset_index(name = 'dau')
)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = dau, 
    x = 'event_date', 
    y = 'dau',
    label = 'DAU', 
    color = 'salmon', 
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Пользователи')
plt.title('DAU за весь период')

plt.show()

In [ ]:
wau = (
    df_events
    .sort_values('event_week')
    .groupby('event_week')['user_id']
    .nunique()
    .reset_index(name = 'wau')
)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = wau, 
    x = 'event_week', 
    y = 'wau',
    label = 'WAU', 
    color = 'salmon', 
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Пользователи')
plt.title('WAU за весь период')

plt.show()

In [ ]:
mau = (
    df_events
    .sort_values('event_month')
    .groupby('event_month')['user_id']
    .nunique()
    .reset_index(name = 'mau')
)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = mau, 
    x = 'event_month', 
    y = 'mau',
    label = 'MAU', 
    color = 'salmon', 
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Пользователи')
plt.title('MAU за весь период')

plt.show()

Метрики показывают устойчивый рост и стабилизацию до начала лета, с июня идёт новый подъём, а затем тенденция на постоянное снижение активности пользователей.

In [ ]:
dau['month_str'] = dau['event_date'].dt.strftime('%Y-%m')

dau_month = (
    dau
    .groupby('month_str')['dau']
    .mean()
    .reset_index(name = 'dau_month')
)

mau['month_str'] = mau['event_month'].dt.strftime('%Y-%m')

stickiness = dau_month.merge(mau, on='month_str')
stickiness['stickiness'] = (stickiness['dau_month'] / stickiness['mau']) * 100
stickiness['date'] = pd.to_datetime(stickiness['month_str'])

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = stickiness, 
    x = 'date', 
    y = 'stickiness',
    label = 'Stickiness', 
    color = 'salmon', 
    marker = 'o',
    linewidth = 2
)
plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Stickiness')
plt.title('Stickiness за весь период')

plt.show()

После роста и небольшой стабилизации `stickiness` показывает постоянный тренд на снижение с июня, особенно резкий спад произощёл в сентябре. Доля возвращающихся пользователей относительно месячной аудитории сокращается.

Построим графики для retention. Товары, которые предлагает маркетплейс, мультикатегорийные, так что посмотрим возвращаемость в течение 4 недель по когортам.

In [ ]:
first_cohort_week = df_events.groupby('user_id')['event_week'].min().reset_index(name = 'cohort_week')

df_ret = df_events.merge(first_cohort_week, on = 'user_id')
df_ret['days_diff'] = (df_ret['event_week'] - df_ret['cohort_week']).dt.days
df_ret['obs_week'] = (df_ret['days_diff'] // 7).clip(upper = 5)

retention_raw = (
    df_ret
    .groupby(['cohort_week', 'obs_week'])['user_id']
    .nunique()
    .reset_index(name = 'users_count')
)

cohort_size = (
    df_ret
    .groupby('cohort_week')['user_id']
    .nunique()
    .reset_index(name = 'total_users')
)

retention_df = retention_raw.merge(cohort_size, on = 'cohort_week', how = 'left')
retention_df['retention_pct'] = (retention_df['users_count'] / retention_df['total_users']) * 100

pivot_ret = retention_df.pivot_table(
    index = 'cohort_week',
    columns = 'obs_week',
    values = 'retention_pct',
    aggfunc = 'first'
)

pivot_ret = pivot_ret.sort_index()
pivot_ret.columns = pivot_ret.columns.astype(int)

cmap = mcolors.LinearSegmentedColormap.from_list('green_red', ['#FF3333', '#FFFF66', '#4C9900'], N = 100)

plt.figure(figsize=(18, 12))

ax = sns.heatmap(
    pivot_ret,
    cmap = cmap,
    annot = True,
    fmt = ".1f",
    linewidths = .5,
    cbar_kws = {'label': 'Retention %'},
    vmin = 0, 
    vmax = 100
)

plt.title('Retention по когортам', fontsize = 16)
plt.xlabel('Неделя')
plt.ylabel('Когорта')

clean_labels = pivot_ret.index.strftime('%Y-%m-%d').tolist()
n_cohorts = len(clean_labels)

if n_cohorts > 20:
    step = 2
    tick_locs = range(0, n_cohorts, step)
    display_labels = clean_labels[::step]
else:
    tick_locs = range(n_cohorts)
    display_labels = clean_labels

ax.set_yticks(tick_locs)
ax.set_yticklabels(display_labels, rotation = 0, fontsize = 10)

plt.tight_layout()
plt.show()

In [ ]:
first_cohort_week = (
    df_events
    .groupby('user_id')['event_week']
    .min()
    .reset_index(name = 'cohort_week')
)

df_ret = df_events.merge(first_cohort_week, on = 'user_id')
df_ret['days_diff'] = (df_ret['event_date'] - df_ret['cohort_week']).dt.days
df_ret = df_ret[df_ret['days_diff'] <= 35]

cohort_size = (
    df_ret
    .groupby('cohort_week')['user_id']
    .nunique()
    .reset_index(name = 'total_users')
)

retained_users = (
    df_ret
    .query('days_diff >= 28 and days_diff <= 30')
    .groupby('cohort_week')['user_id']
    .nunique()
    .reset_index(name = 'retained_users')
)

retention_30d = cohort_size.merge(retained_users, on = 'cohort_week', how  = 'left')
retention_30d['retained_users'] = retention_30d['retained_users'].fillna(0)
retention_30d['retention_30d_pct'] = (retention_30d['retained_users'] / retention_30d['total_users']) * 100
retention_30d = retention_30d.sort_values('cohort_week')

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = retention_30d,
    x = 'cohort_week',
    y = 'retention_30d_pct',
    marker = 'o',
    linewidth = 2,
    color = 'salmon'
)

plt.grid(alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

plt.xlabel('Когорты')
plt.ylabel('Удержание')
plt.title('Скользящее удержание на 30-й день', fontsize = 14)
plt.tight_layout()
plt.show()

До лета удержание было устойчивое. Ранние когорты демонстрируют более высокий Retention, с лета - стабильно ниже. Для e‑commerce такого типа 30‑дневный Retention приблизительно должен быть 5-12%, на графике очень сильное падение. Летом были изменения, после которых ухудшились retention и stickiness.

Посмотрим, как расходуется бюджет на привлечение.

In [ ]:
monthly_spend = df_costs.groupby('spend_month')['budget'].sum().reset_index(name = 'total_budget')
monthly_spend['spend_month'] = pd.to_datetime(monthly_spend['spend_month'])

plt.figure(figsize = (12, 6))

ax = sns.lineplot(
    data = monthly_spend,
    x = 'spend_month',
    y = 'total_budget',
    marker = 'o',
    linewidth = 2, 
    color = 'salmon'
)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
formatter = ScalarFormatter()
formatter.set_scientific(False)
formatter.set_useOffset(False)
ax.yaxis.set_major_formatter(formatter)

plt.grid(axis = 'y', alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

ax.set_title('Распределение бюджета по месяцам', fontsize = 14)
ax.set_xlabel('Даты')
ax.set_ylabel('Бюджет')

plt.tight_layout()
plt.show()

Бюджет на рекламу резко вырос с началом лета - с 1200000 до 1800000. Как распределяется бюджет по каналам привлечения.

In [ ]:
df_costs_channel = df_costs.copy()
df_costs_channel['spend_month'] = pd.to_datetime(df_costs_channel['spend_month'], errors = 'coerce')
df_costs_channel = df_costs_channel.dropna(subset = ['spend_month'])
df_costs_channel = df_costs_channel.sort_values('spend_month')

plt.figure(figsize = (12, 6))

ax = sns.lineplot(
    data = df_costs_channel,
    x = 'spend_month',
    y = 'budget',
    hue = 'acq_channel',
    marker = 'o',
    linewidth = 2
)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

formatter = ScalarFormatter()
formatter.set_scientific(False)
formatter.set_useOffset(False)
ax.yaxis.set_major_formatter(formatter)

plt.grid(axis = 'y', alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

plt.title('Распределение бюджета по каналам привлечения', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('Бюджет')

plt.tight_layout()
plt.show()

Похоже как раз с начала лета произошли изменения в  распределении маркетинго бюджета. На канал `TikTok` траты выросли в разы с июня и постоянно увеличивались, в то же время `Google Ads` резко опустился в приоритете и слился со всеми другими источниками привлечения, которые тоже немного просели с июля. Надо будет узнать, окупились ли такие траты, потому что посещаемость маркетплейса начала стабильно снижаться с очени.

Посчитаем суммарную выручку GMV на каждый месяц (платформа получает 5%).

In [ ]:
monthly_sales = df_orders.groupby('order_month')['total_price'].sum().reset_index()
monthly_sales['marketplace_revenue'] = monthly_sales['total_price'] * 0.05
monthly_sales['marketplace_revenue'] = monthly_sales['marketplace_revenue'].round(2)
monthly_sales = monthly_sales.rename(columns = {'total_price': 'total_sales_gmv'})

plt.figure(figsize = (12, 6))

ax = sns.lineplot(
    data = monthly_sales,
    x = 'order_month',
    y = 'marketplace_revenue',
    marker = 'o',
    linewidth = 2, 
    color = 'salmon'
)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
formatter = ScalarFormatter()
formatter.set_scientific(False)
formatter.set_useOffset(False)
ax.yaxis.set_major_formatter(formatter)

plt.grid(axis = 'y', alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

ax.set_title('Распределение дохода по месяцам', fontsize = 14)
ax.set_xlabel('Даты')
ax.set_ylabel('Доход')

plt.tight_layout()
plt.show()

Месячный доход от маркетплейса показывал стабильный рост до начала лета, затем наступила стагнация и быстрый спад с сентября. Показатель конца года уже ближе к цифрам из начала графика. Направление огромного бюджета на рекламу в тиктоке не помогло увеличить прибыль.

Теперь посмотрим воронку пользователей по уникальным `user_id`.

In [ ]:
steps = ['page_view', 'product_view', 'product_click', 'add_to_cart', 'checkout_start', 'checkout_complete']

df_events_steps = df_events[df_events['event_type'].isin(steps)]
df_events_steps_counts = df_events_steps.groupby('event_type')['user_id'].nunique().reindex(steps)

funnel_df_events = pd.DataFrame({
   'step': df_events_steps_counts.index,
   'users': df_events_steps_counts.values
})

df_events_first_step_users = funnel_df_events.loc[0, 'users']
funnel_df_events['conversion_from_first_%'] = (funnel_df_events['users'] / df_events_first_step_users * 100).round(1)
funnel_df_events['conversion_from_previous_%'] = (funnel_df_events['users'] / funnel_df_events['users'].shift(1) * 100).round(1)

funnel_df_events

In [ ]:
plt.figure(figsize = (12, 6))

plot_df_events = funnel_df_events.iloc[1:]

plt.bar(
    plot_df_events['step'], 
    plot_df_events['conversion_from_first_%'], 
    color = 'salmon'
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.title('Классическая воронка: конверсия от первого шага', fontsize = 14)
plt.ylabel('Конверсия от первого шага')
plt.xlabel('Этапы воронки')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (12, 6))

plot_df_events_p = funnel_df_events.iloc[1:]

plt.bar(
    plot_df_events_p['step'], 
    plot_df_events_p['conversion_from_previous_%'], 
    color = 'salmon'
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.title('Step-by-step воронка: конверсия от предыдущего шага', fontsize = 14)
plt.ylabel('Конверсия от предыдущего шага')
plt.xlabel('Этапы воронки')

plt.tight_layout()
plt.show()

Воронка без критично узких, до последнего этапа доходит больше половины пользователей.


### Итоги обзора

До начала лета метрики продукта росли в стабильной тренде, активность и удержание росли. Прибыль шла вверх. В мае метрики распределялись так: MAU ~ 3250 человек, удержание ~15%, GMV 250000 и расходы на привлечение 1200000.

С началом лета изменился подход к распределению бюджета на привлечение: канал `TikTok` стал получать в разы больше денег, чем все остальные источники, в то же время `Google Ads` опустился ко всем оставшимся каналам ниже 200000. Месячные траты на бюджет увеличились с 1200000 до 1800000. Вместе с этим комиссия продукта стагнировала и начала резко падать с 2500000 в августе до 1300000 в декабре. Врятли на это повлияла сезонность, маркетплейс предоставляет общирные и разноплановые категории товара. К концу года также просели метрики посещений и удержания.

## Оценка метрик монетизации и юнит-экономики

Выполним анализ монетизации. Определим, сколько зарабатывает продукт и сколько тратится на привлечение пользователей. Изучим средний доход, который компания получает от одного пользователя, рассчитаем ARPU.

In [ ]:
df_calc = df_orders.copy()
df_calc['revenue_business'] = df_calc['total_price'] * 0.05
df_calc['order_year_month'] = df_calc['order_date'].dt.to_period('M')

arpu_by_month = (
    df_calc
    .groupby('order_year_month')
    .agg(
        total_revenue = ('revenue_business', 'sum'),
        active_users = ('user_id', 'nunique')
    )
    .reset_index()
)

arpu_by_month['arpu'] = (
    arpu_by_month['total_revenue'] / arpu_by_month['active_users']
).fillna(0).round(2)

arpu_by_month['order_date_plot'] = arpu_by_month['order_year_month'].dt.to_timestamp()

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = arpu_by_month,
    x = 'order_date_plot',
    y = 'arpu',
    marker = 'o',
    linewidth = 2,
    color = 'salmon'
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('ARPU')
plt.title('ARPU за весь период', fontsize = 14)

plt.tight_layout()
plt.show()

ARPU платящего пользователя ~ 1435-1785. Рост был до конца лета, затем произошло снижение в сентябре, попытка выровниться и снова падение к концу года, периоду активных предновогодних продаж. Это согласуется с общим падением дохода, которое мы видели на графике до этого.

Средний чек AOV.

In [ ]:
aov_by_month = (
    df_calc
    .groupby('order_year_month')
    .agg(
        total_revenue = ('revenue_business', 'sum'),
        total_orders = ('order_id', 'nunique')
    )
    .reset_index()
)

aov_by_month['aov'] = (aov_by_month['total_revenue'] / aov_by_month['total_orders']).round(2)
aov_by_month['order_year_month'] = aov_by_month['order_year_month'].apply(lambda x: x.to_timestamp())
aov_by_month = aov_by_month.sort_values('order_year_month')

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = aov_by_month,
    x = 'order_year_month',
    y = 'aov',
    marker = 'o',
    linewidth = 2,
    color = 'salmon'
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('AOV')
plt.title('AOV за весь период', fontsize = 14)

plt.tight_layout()
plt.show()

Был тренд на рост до августа, потом начались резкие передады, возможно из-за предновогодних активностей, распродаж.

### Показатели эффективности

Доработаем датафрейм с профилями пользователей чтобы  оченить эффективность маркетинговых трат.

In [ ]:
df_costs['spend_month'] = pd.to_datetime(df_costs['spend_month'])
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_users['registration_date'] = pd.to_datetime(df_users['registration_date'])

users_count = (
    df_users
    .groupby(['acq_channel', 'cohort_month'])
    .agg(users_in_cohort = ('user_id', 'nunique'))
    .reset_index()
)

costs_with_count = df_costs.merge(
    users_count,
    left_on = ['acq_channel', 'spend_month'],
    right_on = ['acq_channel', 'cohort_month'],
    how = 'left'
)

costs_with_count['cac'] = costs_with_count.apply(
    lambda x: x['budget'] / x['users_in_cohort'] if x['users_in_cohort'] > 0 else 0, 
    axis = 1
)

df_orders_users = df_orders.merge(
    df_users[['user_id', 'registration_date', 'cohort_month', 'acq_channel']],
    on = 'user_id',
    how = 'left'
)

df_orders_users['lifetime'] = (df_orders_users['order_date'] - df_orders_users['registration_date']).dt.days
df_30d = df_orders_users[(df_orders_users['lifetime'] >= 0) & (df_orders_users['lifetime'] < 30)].copy()

commission_rate = 0.05
df_30d['revenue'] = df_30d['total_price'] * commission_rate

user_revenue_30d = (
    df_30d
    .groupby(['user_id', 'acq_channel'])['revenue']
    .sum()
    .reset_index()
)

users_metrics = df_users[['user_id', 'cohort_month', 'acq_channel']].merge(
    user_revenue_30d,
    on = ['user_id', 'acq_channel'],
    how = 'left'
).merge(
    costs_with_count[['acq_channel', 'cohort_month', 'cac']],
    on = ['acq_channel', 'cohort_month'],
    how = 'left'
)

users_metrics['revenue'] = users_metrics['revenue'].fillna(0)

roi_cohort = (
    users_metrics
    .groupby('cohort_month')
    .agg(
        avg_ltv_30d = ('revenue', 'mean'),
        avg_cac = ('cac', 'mean'),
        users_count = ('user_id', 'nunique')
    )
    .reset_index()
)

roi_cohort['roi_percent'] = ((roi_cohort['avg_ltv_30d'] - roi_cohort['avg_cac']) / roi_cohort['avg_cac']) * 100

In [ ]:
roi_cohort

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort, 
    x = 'cohort_month', 
    y = 'avg_cac', 
    marker = 'o',
    linewidth = 2, 
    color = 'salmon'
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('CAC')
plt.title('Средний CAC за весь период', fontsize = 14)

plt.tight_layout()
plt.show()

Затраты на привлечение выросли, но количество пользователей тоже росло со временем, так что подъём получился с ~390 до 405 и относительно стабилен.

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort,
    x = 'cohort_month',
    y = 'avg_ltv_30d',
    marker = 'o',
    linewidth = 2,
    color = 'salmon'
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('LTV')
plt.title('Средний LTV 30 дней', fontsize = 14)

plt.tight_layout()
plt.show()

Выручка на пользователя снижается, это согласуется с падением удержания и возврощаемости. Комиссия упала с января по декабрь в 4 раза.

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort,
    x = 'cohort_month',
    y = 'roi_percent',
    marker = 'o',
    linewidth = 2, 
    color = 'salmon'
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('ROI')
plt.title('Средний ROI 30 дней', fontsize = 14)

plt.tight_layout()
plt.show()

С лета доходность стала отрицательной и постоянно подает.

Посмотрим оценочные сроки окупаемости Payback Period.

In [ ]:
roi_cohort['payback_period_months'] = roi_cohort.apply(
    lambda x: x['avg_cac'] / x['avg_ltv_30d'] if x['avg_ltv_30d'] > 0 else float('inf'),
    axis = 1
)

roi_cohort['payback_period_months'] = roi_cohort['payback_period_months'].round(2)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort,
    x = 'cohort_month',
    y = 'payback_period_months',
    marker = 'o',
    linewidth = 2, 
    color = 'salmon'
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Payback Period')
plt.title('Динамика окупаемости', fontsize = 14)

plt.tight_layout()
plt.show()

Бюджет на привлечение значительно вырос, но относительно увеличения пользователей это нивелируется. Но качество этих пользователей значительно просело - они либо покупают реже, либо меньше тратят, либо вообще не покупают в первые 30 дней. Комиссия с каждым месяцем после весны падает: с ~450 в мае до 150 к ноябрю. Метрика ROI показывает постоянно снижающуюся убыточность с  июня. Срок окупаемости составлял примерно 1 месяц, к концу года уже 2.7 месяца. Решение нагнать трафик из `TikTok` вливанием огромного бюджета в ущерб другим источниками привлечения отразилось на монетизации маркетплейса очень негативным образом (не то что бы `TikTok` плохой канал, может неправильно настроили таргет и т.д.).

## Выводы по этапам 2 и 3

В пераой половине года метрики продукта показывали постепенный органический рост: увеличивались MAU, DAU, удержание в нормальных для маркетплейса такого типа 10-15%, увеличивались выручка, LTV достигал максимума весной. Бюджет на рекламу составлял ~ 1.2 млн ₽ в месяц.

В начале лета (с июня) произошли значительные изменения в маркетинговой системе, траты на рекламу выросли до ~2-2.5 млн ₽, основной упор сделали на канал привлечения `TikTok`, лидирующий до этого канал `Google Ads` практически перестал получать финансирование, все оставшимися трендом идут вниз. Возвращаемость когорт снизилась уже с первой недели: с 45-50% в начале года до 30-40% после июня. LTV поздних когорт хуже. Срок окупаемости растёт.

Модель привлечения стала убыточной. Важные метрики монетизации трендово идут вниз  к концу года, даже не устаканиваются. Рост количества пользователей и прибыль от них не перекрывает траты. Снижение выручки происходит на фоне падения удержания и конверсии, несмотря на рост привлечения. Это указывает на проблему качества трафика.

## Поиск инсайтов, точек роста и сегментация

Перейдём к сегментации и точечным наблюдениям. Посмотрим вглубь данных: по каналам, сегментам пользователей, категориям товаров, регионам, типам устройств.

In [ ]:
mau_os = (
    df_events
    .sort_values('event_month')
    .groupby(['event_month', 'os'])['user_id']
    .nunique()
    .reset_index(name = 'mau')
)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = mau_os, 
    x = 'event_month', 
    y = 'mau',
    hue = 'os',
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Пользователи')
plt.title('MAU по платформам')

plt.show()

In [ ]:
first_visit = (
    df_events
    .groupby('user_id')['event_date']
    .min()
    .reset_index(name = 'first_dt')
)

df_cohort = df_events.merge(first_visit, on = 'user_id', how = 'inner')
df_cohort['age_days'] = (df_cohort['event_date'] - df_cohort['first_dt']).dt.days
df_cohort = df_cohort[df_cohort['age_days'] <= 35]

df_cohort['cohort_week'] = (
    df_cohort['first_dt'].dt.isocalendar().year.astype(str) + '-' + 
    df_cohort['first_dt'].dt.isocalendar().week.astype(str).str.zfill(2)
)

cohort_size = (
    df_cohort
    .groupby(['cohort_week', 'os'])['user_id']
    .nunique()
    .reset_index(name = 'total_users')
)

retained_users = (
    df_cohort
    .query('age_days >= 28 and age_days <= 30')
    .groupby(['cohort_week', 'os'])['user_id']
    .nunique()
    .reset_index(name = 'retained_users')
)

retention_30d = cohort_size.merge(retained_users, on = ['cohort_week', 'os'], how = 'left')
retention_30d['retained_users'] = retention_30d['retained_users'].fillna(0)
retention_30d['retention_30d_pct'] = (retention_30d['retained_users'] / retention_30d['total_users']) * 100

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=retention_30d,
    x = 'cohort_week',
    y = 'retention_30d_pct',
    hue = 'os',
    marker = 'o',
    linewidth = 2.5,
    markersize = 8
)

plt.grid(alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

plt.xlabel('Когорта')
plt.ylabel('Удержание')
plt.title('Скользящее удержание на 30-й день в разрезе платформ', fontsize = 14)

plt.tight_layout()
plt.show()

In [ ]:
mau_device = (
    df_events
    .sort_values('event_month')
    .groupby(['event_month', 'device'])['user_id']
    .nunique()
    .reset_index(name = 'mau')
)

plt.figure(figsize = (12, 6))

sns.lineplot(
    data = mau_device, 
    x = 'event_month', 
    y = 'mau',
    hue = 'device',
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('Пользователи')
plt.title('MAU по устройствам')

plt.show()

In [ ]:
cohort_size = (
    df_cohort
    .groupby(['cohort_week', 'device'])['user_id']
    .nunique()
    .reset_index(name = 'total_users')
)

retained_users = (
    df_cohort
    .query('age_days >= 28 and age_days <= 30')
    .groupby(['cohort_week', 'device'])['user_id']
    .nunique()
    .reset_index(name = 'retained_users')
)

retention_30d = cohort_size.merge(retained_users, on = ['cohort_week', 'device'], how = 'left')
retention_30d['retained_users'] = retention_30d['retained_users'].fillna(0)
retention_30d['retention_30d_pct'] = (retention_30d['retained_users'] / retention_30d['total_users']) * 100

plt.figure(figsize=(12, 6))

sns.lineplot(
    data = retention_30d,
    x = 'cohort_week',
    y = 'retention_30d_pct',
    hue = 'device',
    marker = 'o',
    linewidth = 2.5,
    markersize = 8
)

plt.grid(alpha = 0.4, linestyle = '--')
plt.xticks(rotation = 45, fontsize = 10)

plt.xlabel('Даты')
plt.ylabel('Удержание')
plt.title('Скользящее удержание на 30-й день в разрезе устройств', fontsize = 14)

plt.tight_layout()
plt.show()

Значимой разницы по устройствам и платформам нет. Изучим воронку маркетплейса детальнее.

In [ ]:
mobile_df_events = df_events[df_events['device'] == 'mobile']
tablet_df_events = df_events[df_events['device'] == 'tablet']
desktop_df_events = df_events[df_events['device'] == 'desktop']

counts_mobile = mobile_df_events[mobile_df_events['event_type'].isin(steps)]
counts_mobile_agg = counts_mobile.groupby('event_type')['user_id'].nunique().reindex(steps)
mobile_funnel = (counts_mobile_agg / counts_mobile_agg.shift(1) * 100).round(1)

counts_tablet = tablet_df_events[tablet_df_events['event_type'].isin(steps)]
counts_tablet_agg = counts_tablet.groupby('event_type')['user_id'].nunique().reindex(steps)
tablet_funnel = (counts_tablet_agg / counts_tablet_agg.shift(1) * 100).round(1)

counts_desktop = desktop_df_events[desktop_df_events['event_type'].isin(steps)]
counts_desktop_agg = counts_desktop.groupby('event_type')['user_id'].nunique().reindex(steps)
desktop_funnel = (counts_desktop_agg / counts_desktop_agg.shift(1) * 100).round(1)

funnel_compare_df_events = pd.DataFrame({
    'step': steps[1:],
    'mobile': mobile_funnel.iloc[1:].values,
    'tablet': tablet_funnel.iloc[1:].values,
    'desktop': desktop_funnel.iloc[1:].values
})

funnel_compare_df_events

In [ ]:
windows_df_events = df_events[df_events['os'] == 'windows']
android_df_events = df_events[df_events['os'] == 'android']
ios_df_events = df_events[df_events['os'] == 'ios']
macos_df_events = df_events[df_events['os'] == 'macos']

counts_windows = windows_df_events[windows_df_events['event_type'].isin(steps)]
counts_windows_agg = counts_windows.groupby('event_type')['user_id'].nunique().reindex(steps)
windows_funnel = (counts_windows_agg / counts_windows_agg.shift(1) * 100).round(1)

counts_android = android_df_events[android_df_events['event_type'].isin(steps)]
counts_android_agg = counts_android.groupby('event_type')['user_id'].nunique().reindex(steps)
android_funnel = (counts_android_agg / counts_android_agg.shift(1) * 100).round(1)

counts_ios = ios_df_events[ios_df_events['event_type'].isin(steps)]
counts_ios_agg = counts_ios.groupby('event_type')['user_id'].nunique().reindex(steps)
ios_funnel = (counts_ios_agg / counts_ios_agg.shift(1) * 100).round(1)

counts_macos = macos_df_events[macos_df_events['event_type'].isin(steps)]
counts_macos_agg = counts_macos.groupby('event_type')['user_id'].nunique().reindex(steps)
macos_funnel = (counts_macos_agg / counts_macos_agg.shift(1) * 100).round(1)

funnel_compare_df_events_os = pd.DataFrame({
    'step': steps[1:],
    'windows': windows_funnel.iloc[1:].values,
    'android': android_funnel.iloc[1:].values,
    'ios': ios_funnel.iloc[1:].values,
    'macos': macos_funnel.iloc[1:].values
})

funnel_compare_df_events_os

Разницы в воронках по устройствам и платформам почти нет. Изучим метрики монетизации в разрезе каналов привлечения.

In [ ]:
df_costs['spend_month'] = pd.to_datetime(df_costs['spend_month']).dt.to_period('M')
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
df_users['registration_date'] = pd.to_datetime(df_users['registration_date'])
df_users['cohort_month'] = df_users['registration_date'].dt.to_period('M')

users_count = (
    df_users
    .groupby(['acq_channel', 'cohort_month'])
    .agg(users_in_cohort = ('user_id', 'nunique'))
    .reset_index()
)

df_costs['spend_month_period'] = df_costs['spend_month']

costs_with_count = df_costs.merge(
    users_count,
    left_on = ['acq_channel', 'spend_month_period'],
    right_on = ['acq_channel', 'cohort_month'],
    how = 'left'
)

costs_with_count['cac'] = costs_with_count.apply(
    lambda x: x['budget'] / x['users_in_cohort'] if x['users_in_cohort'] > 0 else np.nan, 
    axis = 1
)

cac_summary = (
    costs_with_count
    .groupby(['acq_channel', 'cohort_month'])['cac']
    .mean()
    .reset_index()
)

df_orders_users = df_orders.merge(
    df_users[['user_id', 'registration_date', 'cohort_month', 'acq_channel']],
    on = 'user_id',
    how = 'left'
)

df_orders_users['lifetime'] = (df_orders_users['order_date'] - df_orders_users['registration_date']).dt.days
df_30d = df_orders_users[(df_orders_users['lifetime'] >= 0) & (df_orders_users['lifetime'] < 30)].copy()

commission_rate = 0.05
df_30d['revenue'] = df_30d['total_price'] * commission_rate

user_revenue_30d = (
    df_30d
    .groupby(['user_id', 'acq_channel'])['revenue']
    .sum()
    .reset_index()
)

users_metrics = df_users[['user_id', 'cohort_month', 'acq_channel']].merge(
    user_revenue_30d,
    on = ['user_id', 'acq_channel'],
    how = 'left'
)

users_metrics = users_metrics.merge(
    cac_summary,
    on = ['acq_channel', 'cohort_month'],
    how = 'left'
)


users_metrics['revenue'] = users_metrics['revenue'].fillna(0)

roi_cohort_by_channel = (
    users_metrics
    .groupby(['cohort_month', 'acq_channel'])
    .agg(
        avg_ltv_30d = ('revenue', 'mean'),
        avg_cac = ('cac', 'mean'),
        users_count = ('user_id', 'nunique'),
        zero_buyers = ('revenue', lambda x: (x == 0).sum())
    )
    .reset_index()
)

roi_cohort_by_channel['payback_period'] = roi_cohort_by_channel.apply(
    lambda x: x['avg_cac'] / x['avg_ltv_30d'] if x['avg_ltv_30d'] > 0 else np.inf, 
    axis = 1
)

roi_cohort_by_channel['roi_percent'] = ((roi_cohort_by_channel['avg_ltv_30d'] - roi_cohort_by_channel['avg_cac']) / roi_cohort_by_channel['avg_cac']) * 100
roi_cohort_by_channel['zero_buyers_rate'] = (roi_cohort_by_channel['zero_buyers'] / roi_cohort_by_channel['users_count']) * 100
roi_cohort_by_channel = roi_cohort_by_channel.sort_values(['cohort_month', 'acq_channel'])
roi_cohort_by_channel['cohort_date'] = roi_cohort_by_channel['cohort_month'].dt.to_timestamp()

roi_cohort_by_channel.head()

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort_by_channel, 
    x = 'cohort_date', 
    y = 'avg_cac', 
    hue = 'acq_channel',
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('CAC')
plt.title('Средний CAC по каналам', fontsize = 14)

plt.tight_layout()
plt.show()

Стоимость привлечения внутри каждого канала в течение 2024 года была относительно стабильной.

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort_by_channel, 
    x = 'cohort_date', 
    y = 'avg_ltv_30d', 
    hue = 'acq_channel',
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.xlabel('Даты')
plt.ylabel('LTV')
plt.title('Средний LTV по каналам 30 дней', fontsize = 14)

plt.tight_layout()
plt.show()

Больше всего денег приносят `Affiliate` `Email` `Google Ads` и `SEO`, у них стабильная динамика, графики постепенно растут. Самый слабый канал - `TikTok`.

In [ ]:
plt.figure(figsize = (12, 6))

sns.lineplot(
    data = roi_cohort_by_channel, 
    x = 'cohort_date', 
    y = 'roi_percent', 
    hue = 'acq_channel',
    marker = 'o',
    linewidth = 2
)

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)

plt.xlabel('Недели')
plt.ylabel('ROI')
plt.title('Средний ROI по каналам 30 дней', fontsize = 14)

plt.tight_layout()
plt.show()

Эффективность каналов привлечения очень разная. Стабильно выше 0 метрика ROI только у `Affiliate` и `SEO`. Еле еле несколько месяцев в году выше 0 каналы `Google Ads` и `Social Media`. Наиболее низкий ROI у `TikTok`, при этом расходы на него существенно выросли с лета. Возможно, этот канал мог способствовать снижению общей эффективности маркетинговых инвестиций.

In [ ]:
budget_by_channel = (
    df_costs
    .groupby('acq_channel')['budget']
    .sum()
    .reset_index(name = 'total_budget')
)

users_by_channel = (
    df_users
    .groupby('acq_channel')['user_id']
    .nunique()
    .reset_index(name = 'users_count')
)

channel_summary = budget_by_channel.merge(
    users_by_channel,
    on = 'acq_channel',
    how = 'left'
)

channel_summary['budget_share'] = (
    channel_summary['total_budget'] /
    channel_summary['total_budget'].sum() * 100
)

channel_summary['users_share'] = (
    channel_summary['users_count'] /
    channel_summary['users_count'].sum() * 100
)

channel_summary['budget_users_ratio'] = (
    channel_summary['budget_share'] /
    channel_summary['users_share']
)

channel_table = channel_summary[['acq_channel', 'budget_share', 'users_share', 'budget_users_ratio']].copy()

channel_table['budget_share'] = channel_table['budget_share'].round(1)
channel_table['users_share'] = channel_table['users_share'].round(1)
channel_table['budget_users_ratio'] = channel_table['budget_users_ratio'].round(2)

channel_table

Рост стоимости привлечения не коррелирует с качеством пользователей. `TikTok` приводит много пользователей, но они не монетизируются, при этом ест половину маркетингового бюджета. Каналы `Affiliate` и `SEO` получают меньшую долю бюджета, но показывают хороший средний LTV относительно CAC.

In [ ]:
user_metrics = df_users[['user_id', 'cohort_month', 'acq_channel']].copy()
buyers = set(df_orders['user_id'])
user_metrics['is_buyer'] = user_metrics['user_id'].isin(buyers)

conversion_by_channel = (
    user_metrics
    .groupby('acq_channel')
    .agg(
        users = ('user_id', 'nunique'),
        buyers = ('is_buyer', 'sum')
    )
    .reset_index()
)

conversion_by_channel['conversion'] = (
    conversion_by_channel['buyers'] /
    conversion_by_channel['users'] * 100
)

conversion_by_channel

`TikTok` приводит больше всех пользователей, которые практически не конвертируются в покупателей. Почти все отваливаются на каком-то этапе.

In [ ]:
df_events_channel = df_events.merge(
    df_users[['user_id', 'acq_channel']],
    on = 'user_id',
    how = 'left'
)

df_tiktok_events = df_events_channel[
    (df_events_channel['acq_channel'] == 'TikTok') &
    (df_events_channel['event_type'].isin(steps))
]

df_tiktok_counts = (
    df_tiktok_events
    .groupby('event_type')['user_id']
    .nunique()
    .reindex(steps)
)

funnel_tiktok = pd.DataFrame({
    'step': df_tiktok_counts.index,
    'users': df_tiktok_counts.values
})

first_users = funnel_tiktok.loc[0, 'users']

funnel_tiktok['conversion_from_first_%'] = (
    funnel_tiktok['users'] / first_users * 100
).round(1)

funnel_tiktok['conversion_from_previous_%'] = (
    funnel_tiktok['users'] /
    funnel_tiktok['users'].shift(1) * 100
).round(1)

funnel_tiktok

In [ ]:
plt.figure(figsize = (12, 6))

plot_df_funnel_tiktok = funnel_tiktok.iloc[1:]

plt.bar(
    plot_df_funnel_tiktok['step'], 
    plot_df_funnel_tiktok['conversion_from_first_%'], 
    color = 'salmon'
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.title('Классическая воронка: конверсия от первого шага для канала ТикТок', fontsize = 14)
plt.ylabel('Конверсия от первого шага')
plt.xlabel('Этапы воронки')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (12, 6))

plot_df_funnel_tiktok_p = funnel_tiktok.iloc[1:]

plt.bar(
    plot_df_funnel_tiktok_p['step'], 
    plot_df_funnel_tiktok_p['conversion_from_previous_%'], 
    color = 'salmon'
)

plt.grid(alpha = .4)
plt.xticks(rotation = 45)

plt.title('Step-by-step воронка: конверсия от предыдущего шага для канала ТикТок', fontsize = 14)
plt.ylabel('Конверсия от предыдущего шага')
plt.xlabel('Этапы воронки')

plt.tight_layout()
plt.show()

Чекаут - самое слабое место.

In [ ]:
events_with_channel = df_events.merge(df_users[['user_id', 'acq_channel']], on = 'user_id', how = 'left')

comparison = (
    events_with_channel
    .groupby(['acq_channel', 'event_type'])['user_id']
    .nunique()
    .reset_index(name = 'users')
)

channel_users = (
    df_users
    .groupby('acq_channel')['user_id']
    .nunique()
    .reset_index(name = 'channel_users')
)

comparison = comparison.merge(channel_users,  on = 'acq_channel', how = 'left')
comparison['user_share_%'] = (comparison['users'] / comparison['channel_users'] * 100).round(1)

behavior_comparison = comparison[
    comparison['event_type'].isin([
        'search',
        'filter_apply',
        'category_view',
        'wishlist_add',
        'banner_click',
        'promo_view'
    ])].pivot(index = 'event_type', columns = 'acq_channel', values = 'user_share_%'
).round(1)

behavior_comparison

Пользователи, привлечённые из `TikTok`, хуже всего показывают себя в дополнительных действиях. Они не только реже доходят до чекаута, но и слабо исследуют маркетплейс. 

Поищем другие инсайты.

In [ ]:
segment_channel = pd.crosstab(
    df_users['acq_channel'],
    df_users['buyer_segment'],
    normalize='index'
) * 100

segment_channel

In [ ]:
bins = [0, 24, 34, 44, 54, 100]
labels = ['18-24', '25-34', '35-44', '45-54', '55+']

df_users['age_group'] = pd.cut(
    df_users['age'],
    bins = bins,
    labels = labels
)

age_channel = pd.crosstab(
    df_users['acq_channel'],
    df_users['age_group'],
    normalize = 'index'
) * 100

age_channel

In [ ]:
region_channel = pd.crosstab(
    df_users['region'],
    df_users['acq_channel'],
    normalize = 'index'
) * 100

region_channel

In [ ]:
gender_channel = pd.crosstab(
    df_users['acq_channel'],
    df_users['gender'],
    normalize = 'index'
) * 100

gender_channel

Разница по группам и каналам совсем небольшая - 1-2%.

In [ ]:
orders_users = df_orders.merge(
    df_users[['user_id', 'age', 'gender', 'region', 'acq_channel', 'buyer_segment']],
    on = 'user_id',
    how = 'left'
)

segment_revenue = (
    orders_users
    .groupby('buyer_segment')
    .agg(
        users = ('user_id', 'nunique'),
        orders = ('order_id', 'nunique'),
        revenue = ('total_price', 'sum')
    )
    .reset_index()
)

segment_revenue['revenue_share_%'] = (segment_revenue['revenue'] / segment_revenue['revenue'].sum() * 100).round(1)
segment_revenue

Меньше всего людей останавливаются на одной покупке. Пользователи, покупающие регулярно, приносят две трети дохода. 

In [ ]:
orders_users['age_group'] = pd.cut(orders_users['age'], bins = bins, labels = labels)

age_revenue = (
    orders_users
    .groupby('age_group', observed=True)
    .agg(
        users = ('user_id', 'nunique'),
        orders = ('order_id', 'nunique'),
        revenue = ('total_price', 'sum')
    )
    .reset_index()
)

age_revenue['revenue_share_%'] = (age_revenue['revenue'] /age_revenue['revenue'].sum() * 100).round(1)

age_revenue

Самая слабая возрастная группа - до 24 лет.

In [ ]:
category_revenue = (
    df_orders
    .groupby('category_name')
    .agg(
        orders = ('order_id', 'nunique'),
        buyers = ('user_id', 'nunique'),
        quantity = ('quantity', 'sum'),
        revenue = ('total_price', 'sum')
    )
    .reset_index()
)

category_revenue['revenue_share_%'] = (category_revenue['revenue'] / category_revenue['revenue'].sum() * 100).round(1)

category_revenue

Самые успешные категории: мебель для дома, бытовая техника, украшения и часы.

In [ ]:
top_products = (
    df_orders
    .groupby('product_name')
    .agg(
        orders = ('order_id', 'nunique'),
        buyers = ('user_id', 'nunique'),
        quantity = ('quantity', 'sum'),
        revenue = ('total_price', 'sum')
    )
    .reset_index()
    .sort_values('revenue', ascending = False)
)

top_products.head(15)

Мебель, украшения, бытовая техника - самые популярные товары. 

In [ ]:
age_by_channel = pd.crosstab(
    df_users['age_group'],
    df_users['acq_channel'],
    normalize = 'columns'
) * 100

age_by_channel

Получается распределение по возрастным группам равномерное для всех каналов привлечения, и оно перевешивает в сторону более возрастных потребителей. Самые прибыльные товары конечно дорогие, но может одежда/обувь/книги/декор/канцтовары и т.д. как раз и не растут в доходах из-за странного таргета аудитории. Украшения и часы, например, составляют крупную долю в прибыли.

`TikTok` приводит большой объём пользователей, но большая часть из них не становится активными платящими клиентами. Бюджет распределён не пропорционально эффективности каналов. Возможно нужно перераспределение бюджета в пользу каналов с более высоким LTV/CAC, например `Affiliate` и `SEO`. `Google Ads` работал не так плохо в первой половину года. `Social Media` находится около точки безубыточности. 

## Выводы по этапу 5

Можно предположить, что увеличение маркетинговых инвестиций во второй половине 2014 года не привело к пропорциональному росту качества привлечённых пользователей и не повысило прибыль с комиссии.

Самый проблемный канал привлечения - `TikTok`. Привлекает много аудитории, но она слабо конвертируется в покупки и приносит существенно меньше выручки. Лучше всего работают `Affiliate`, `SEO` и `Google Ads`. У всех каналов есть перекос на более возрастную аудиторию, возможно более точечное таргетирование с разделением по  вазрасту и интересам привело бы к более высокой конвертации в продукт.

Наиболее значимый сегмент по выручке — regular, это основной источник монетизации. Можно изучить, как переводить больше пошльзователей в эту группу. Наибольшую долю выручки формируют: мебель для дома, бытовая техника, украшения и часы, спортивный инвентарь и товары для туризма.


Гипотезы:
- Если перераспределить часть рекламного бюджета `TikTok` на `Email Marketing`, `Google Ads` и `Social Media`, то окупаемость затрат на привлечение и удержание пользователей вырастет (сейчас `TikTok` самый неэффективный канал по соотношению затрат/выгоды, а каналы `Email Marketing`, `Google Ads` и `Social Media` находтся около линии окупаемости).
- Если масштабировать и увеличить бюджет для каналов `Affiliate` и `SEO` за счёт менее эффективных источников, то можно увеличить объём выручки без пропорционального роста затрат.
- Если уделить дополнительное внимание `medium` пользователям (персонализированные рекомендаций, скидки, акции), то они вероятнее перейдут в группу `regular` и принесут больше комиссии платформе.
- Если скорректировать таргет рекламных компаний и источников привлечения на более молодую аудиторию + категории товара мелкого и среднего сегмента, то можно увеличить конвертацию в покупку у пользователей до 35 лет и повысить возвращаемость новых когорт.
-------

## Подготовка эксперимента и подведение его результатов

Команда продукта рассмотрела гипотезы и решила влиять на конверсию при помощи финансовых стимуляций. Так как ситуация со снижением выручки срочная и безотлагательная, подготовили дизайн документа, проведём эксперимент.

Загрузим данные с результатами эксперимента, проверим корректность теста.

In [ ]:
df_orders_ab = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_orders_AB.csv', parse_dates = ['order_date','order_week','order_month'])
df_split_ab = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_AB_split_users.csv')
df_users_ab = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_users_AB.csv', parse_dates = ['registration_date','cohort_week','cohort_month'])
df_events_ab = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_events_AB.csv', parse_dates = ['event_date','event_week','event_month'])

In [ ]:
df_split_ab.head()

In [ ]:
df_split_ab.info()

In [ ]:
df_split_ab['group'].value_counts()

In [ ]:
df_abt_group_cont = df_split_ab[df_split_ab['group'] == 'control']['user_id'].unique()
df_abt_group_test = df_split_ab[df_split_ab['group'] == 'treatment']['user_id'].unique()

df_abt_inter = set(df_abt_group_cont).intersection(set(df_abt_group_test))
df_abt_inter

Пересечений групп пользователей нет.

In [ ]:
df_split_ab.duplicated().sum()

In [ ]:
df_split_ab['user_id'].nunique()

In [ ]:
test_group = df_split_ab.groupby('group')['user_id'].nunique()
test_group

part_group_cont = test_group['control']
part_group_test = test_group['treatment']

P = round(100 * abs(part_group_cont - part_group_test) / part_group_cont, 2)
P

Разница численности групп меньше 3%.

In [ ]:
df_users_ab.head()

In [ ]:
df_users_ab.info()

In [ ]:
df_users_ab['acq_channel'].value_counts()

In [ ]:
df_users_ab_test = df_split_ab.merge(
    df_users_ab,
    on = 'user_id',
    how = 'left',
    validate = 'one_to_one'
)

In [ ]:
df_users_ab_test.head()

In [ ]:
df_users_ab_test.info()

In [ ]:
df_users_ab_test['cohort_week'].min()

In [ ]:
df_users_ab_test['cohort_week'].max()

In [ ]:
df_users_ab_test['registration_date'].min()

In [ ]:
df_users_ab_test['registration_date'].max()

In [ ]:
df_orders_ab.head()

In [ ]:
df_orders_ab.info()

In [ ]:
df_orders_ab_test = df_orders_ab.merge(
    df_split_ab,
    on = 'user_id',
    how = 'inner',
    validate = 'many_to_one'
)

In [ ]:
df_orders_ab_test.head()

In [ ]:
users_in_orders_after_merge = df_orders_ab_test['user_id'].nunique()
users_in_orders_after_merge

Заказы совершили только 126 пользователей из 2968.

In [ ]:
df_orders_ab_test.info()

In [ ]:
df_orders_ab_test['order_date'].min()

In [ ]:
df_orders_ab_test['order_date'].max()

In [ ]:
df_orders_ab_test['category_name'].value_counts()

In [ ]:
df_orders_ab_test['product_name'].value_counts()

In [ ]:
df_orders_ab_test['group'].value_counts()

In [ ]:
df_orders_ab_test['unit_price'].describe()

In [ ]:
df_orders_ab_test['total_price'].describe()

In [ ]:
df_events_ab.head()

In [ ]:
df_events_ab.info()

In [ ]:
df_test_events_test = df_events_ab.merge(
    df_split_ab,
    on = 'user_id',
    how = 'inner',
    validate = 'many_to_one'
)

In [ ]:
df_test_events_test.head()

In [ ]:
df_test_events_test.info()

In [ ]:
df_test_events_test['user_id'].value_counts()

In [ ]:
df_test_events_test['event_type'].value_counts()

In [ ]:
df_test_events_test['product_name'].value_counts()

In [ ]:
df_test_events_test['device'].value_counts()

In [ ]:
df_test_events_test['os'].value_counts()

Основная единица анализа — пользователь. Убедиться в том, что их распределение по группам теста не искажает другие показатели.

In [ ]:
df_users_ab_test['age_group'] = pd.cut(
    df_users_ab_test['age'],
    bins = bins,
    labels = labels
)

plat_cont = (
    df_users_ab_test[df_users_ab_test['group'] == 'control']
    [['user_id', 'age_group']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('age_group', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

plat_test = (
    df_users_ab_test[df_users_ab_test['group'] == 'treatment']
    [['user_id', 'age_group']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('age_group', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

sizes_cont = plat_cont['count'].values
labels_cont = plat_cont['age_group'].values
sizes_test = plat_test['count'].values
labels_test = plat_test['age_group'].values

fig, (c, t) = plt.subplots(1, 2, figsize = (15, 8))

c.pie(
    sizes_cont,
    labels = labels_cont,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue', 'lightgreen', 'lightyellow', 'lightpink']
)

t.pie(
    sizes_test,
    labels = labels_test,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue', 'lightgreen', 'lightyellow', 'lightpink']
)

plt.suptitle('Сравнение долевого разделения групп по возрасту', fontsize = 14)
c.set_title('Контрольная группа')
t.set_title('Тестовая группа')

plt.tight_layout()
plt.show()

In [ ]:
segment_cont = (
    df_users_ab_test[df_users_ab_test['group'] == 'control']
    [['user_id', 'buyer_segment']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('buyer_segment', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

segment_test = (
    df_users_ab_test[df_users_ab_test['group'] == 'treatment']
    [['user_id', 'buyer_segment']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('buyer_segment', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

sizes_cont = segment_cont['count'].values
labels_cont = segment_cont['buyer_segment'].values
sizes_test = segment_test['count'].values
labels_test = segment_test['buyer_segment'].values

fig, (c, t) = plt.subplots(1, 2, figsize=(15, 8))

c.pie(
    sizes_cont,
    labels = labels_cont,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue', 'lightgreen', 'lightyellow']
)

t.pie(
    sizes_test,
    labels = labels_test,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue', 'lightgreen', 'lightyellow']
)

plt.suptitle('Сравнение долевого разделения групп по сегменту', fontsize = 14)
c.set_title('Контрольная группа')
t.set_title('Тестовая группа')

plt.tight_layout()
plt.show()

In [ ]:
gender_cont = (
    df_users_ab_test[df_users_ab_test['group'] == 'control']
    [['user_id', 'gender']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('gender', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

gender_test = (
    df_users_ab_test[df_users_ab_test['group'] == 'treatment']
    [['user_id', 'gender']]
    .drop_duplicates(subset = ['user_id'])
    .groupby('gender', as_index = False)['user_id']
    .nunique()
    .rename(columns = {'user_id': 'count'})
)

sizes_cont = gender_cont['count'].values.astype(float)
labels_cont = gender_cont['gender'].values
sizes_test = gender_test['count'].values.astype(float)
labels_test = gender_test['gender'].values

fig, (c, t) = plt.subplots(1, 2, figsize = (15, 8))

c.pie(
    sizes_cont,
    labels = labels_cont,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue']
)

t.pie(
    sizes_test,
    labels = labels_test,
    autopct = '%1.1f%%',
    colors = ['salmon', 'lightblue']
)

plt.suptitle('Сравнение долевого разделения групп по гендеру', fontsize = 14)
c.set_title('Контрольная группа')
t.set_title('Тестовая группа')

plt.tight_layout()
plt.show()

Разница в группах по разным срезам до 5%, это приемлемо. Можно сказать, что тест проходит корректно. Различие в количестве пользователей в двух группах незначительное. Выборки независимые, пересечений обнаружено не было. Тест проводится 3 недели.

### Сравнение воронок событий.

Сравним пользовательский путь новых пользователей в тестовой и контрольной группах. Построим классическую и последовательную воронки, выполним визуализацию.

In [ ]:
df_test_events_test_cont = df_test_events_test[df_test_events_test['group'] == 'control']
df_test_events_test_test = df_test_events_test[df_test_events_test['group'] == 'treatment']

In [ ]:
df_hist_steps_cont = df_test_events_test_cont[df_test_events_test_cont['event_type'].isin(steps)]
df_hist_steps_counts_cont = df_hist_steps_cont.groupby('event_type')['user_id'].nunique().reindex(steps)

funnel_df_hist_cont = pd.DataFrame({
   'step': df_hist_steps_counts_cont.index,
   'users': df_hist_steps_counts_cont.values
})

df_hist_first_step_users_cont = funnel_df_hist_cont.loc[0, 'users']
funnel_df_hist_cont['conversion_from_first_%'] = (funnel_df_hist_cont['users'] / df_hist_first_step_users_cont * 100).round(1)
funnel_df_hist_cont['conversion_from_previous_%'] = (funnel_df_hist_cont['users'] / funnel_df_hist_cont['users'].shift(1) * 100).round(1)

funnel_df_hist_cont

In [ ]:
df_hist_steps_test = df_test_events_test_test[df_test_events_test_test['event_type'].isin(steps)]
df_hist_steps_counts_test = df_hist_steps_test.groupby('event_type')['user_id'].nunique().reindex(steps)

funnel_df_hist_test = pd.DataFrame({
   'step': df_hist_steps_counts_test.index,
   'users': df_hist_steps_counts_test.values
})

df_hist_first_step_users_test = funnel_df_hist_test.loc[0, 'users']
funnel_df_hist_test['conversion_from_first_%'] = (funnel_df_hist_test['users'] / df_hist_first_step_users_test * 100).round(1)
funnel_df_hist_test['conversion_from_previous_%'] = (funnel_df_hist_test['users'] / funnel_df_hist_test['users'].shift(1) * 100).round(1)

funnel_df_hist_test

In [ ]:
plt.figure(figsize = (12, 6))

plt.plot(
    funnel_df_hist_cont['step'],
    funnel_df_hist_cont['conversion_from_previous_%'],
    marker = 'o',
    linestyle = '-',
    color = 'salmon',
    linewidth = 2,
    label = 'control'
)

plt.plot(
    funnel_df_hist_test['step'],
    funnel_df_hist_test['conversion_from_previous_%'],
    marker = 's',
    linestyle = '-',
    color = 'lightblue',
    linewidth = 2,
    label = 'treatment'
)

plt.title('Step-by-step воронка: конверсия от предыдущего шага в разрезе тестовых групп', fontsize = 14)
plt.xlabel('Этапы воронки')
plt.ylabel('Конверсия от предыдущего шага %')

plt.legend(fontsize = 12)
plt.grid(True, linestyle = '--', alpha = 0.5)
plt.xticks(rotation = 45, ha = 'right')

plt.tight_layout()
plt.show()

В тестовой группе конверсия воронки намного лучше, чем в контрольной. Количество пользователей, которые доходят до этапа чекаута выше на 6%, а заканчивают покупку на 4% больше людей.

### Дизайн эксперимента: Бонус на первую покупку для пользователей TikTok

Бизнес-контекст: Команда продукта запускает эксперимент, направленный на повышение конверсии среди пользователей, привлечённых через канал TikTok. Для стимулирования первой покупки пользователям будет предоставлен бонус на первую покупку.

Цель: Проверить, влияет ли предоставление бонуса на первую покупку на конверсию и выручку среди пользователей TikTok.

Гипотеза: Предоставление бонуса на первую покупку новым пользователям, привлеченным через TikTok, приведет к увеличению конверсии в первую покупку.

**Формат эксперимента**

Тип эксперимента: Сплит-роллинг (rolling split). Канал: Только пользователи, пришедшие с TikTok. Группы: Контрольная (A): пользователи TikTok без бонуса, тестовая (B): пользователи TikTok с бонусом на первую покупку. Распределение: Равномерное распределение новых пользователей TikTok между группами по дате регистрации. Период проведения: 3 недели (фиксировано)

**Ключевые метрики**

Основная метрика: Конверсия в первую покупку — доля пользователей, зарегистрировавшихся с TikTok и совершивших первую покупку. Цель: увеличить конверсию с помощью бонуса. Критерий успеха: статистически значимый рост конверсии в тестовой группе.

Вспомогательные метрики: ARPU (средняя выручка на пользователя TikTok). ARPPU (средняя выручка на покупателя TikTok). AOV (средний чек TikTok).

Барьерная метрика: Общая выручка от пользователей TikTok.

**Размеры групп и запуск**

Целевая аудитория: только пользователи TikTok. 
Размеры групп: определяются по фактическому трафику TikTok за период эксперимента.
Комментарий: запуск без предварительного расчёта мощности и MDE из-за срочности задачи.

----

Начнём с основной метрики. Будем использовать z-тест для долей конверсии, уровень значимости стандартный α = 0.05 и сделаем поправку на окно в 3 недели.

H0 - Предоставление бонуса на первую покупку новым пользователям, привлеченным через TikTok, не приведет к увеличению конверсии в первую покупку.

H1 - Предоставление бонуса на первую покупку новым пользователям, привлеченным через TikTok, приведет к увеличению конверсии в первую покупку.

In [ ]:
ab_users = df_split_ab['user_id'].unique()
df_abt = df_users_ab[df_users_ab['user_id'].isin(ab_users)].copy()

df_abt = df_abt.merge(
    df_split_ab[['user_id', 'group']],
    on = 'user_id',
    how = 'left',
    validate=  'one_to_one'
)

orders = df_orders_ab[df_orders_ab['user_id'].isin(ab_users)].copy()

orders = orders.merge(
    df_split_ab[['user_id', 'group']],
    on = 'user_id',
    how = 'left',
    validate = 'many_to_one'
)

orders = orders.merge(
    df_abt[['user_id', 'registration_date']],
    on = 'user_id',
    how = 'left',
    validate = 'many_to_one'
)

orders['days_after_registration'] = (orders['order_date'] - orders['registration_date']).dt.days
orders_21d = orders[(orders['days_after_registration'] >= 0) & (orders['days_after_registration'] <= 20)].copy()

first_orders = (
    orders_21d
    .groupby('user_id')
    .agg(first_purchase_ts = ('order_date', 'min'))
    .reset_index()
)

cohort = df_abt[
    ['user_id', 'group', 'registration_date']
].merge(
    first_orders[['user_id', 'first_purchase_ts']],
    on = 'user_id',
    how = 'left'
)

cohort['converted'] = (cohort['first_purchase_ts'].notna()).astype(int)
cohort['cohort_date'] = cohort['registration_date'].dt.date

agg = (
    cohort
    .groupby('group')['converted']
    .agg(['sum', 'count'])
    .rename(
        columns = {
            'sum': 'conversions',
            'count': 'total_users'
        }
    )
)

agg['conversion_rate'] = (agg['conversions'] / agg['total_users'])
count = agg.loc[['treatment', 'control'], 'conversions'].values
nobs = agg.loc[['treatment', 'control'], 'total_users'].values

z_stat, p_val = proportions_ztest(
    count,
    nobs,
    alternative = 'larger'
)

conv_control = agg.loc['control', 'conversion_rate']
conv_treatment = agg.loc['treatment', 'conversion_rate']

absolute_lift = (conv_treatment - conv_control)
relative_lift = (100 * (conv_treatment / conv_control - 1) if conv_control != 0 else 0)

results_df = pd.DataFrame([{
    'Metric': 'Key',
    'Group A': 'control',
    'Group B': 'treatment',
    'p-value': p_val,
    'Conv A': conv_control,
    'Conv B': conv_treatment,
    'Absolute Lift': absolute_lift,
    'Relative Lift (%)': relative_lift
}])

results_df

p‑value показывает, что разница между группами теста статистически значима.

In [ ]:
records = []

cohort['first_purchase_date'] = cohort['first_purchase_ts'].dt.date
dates = sorted(cohort['cohort_date'].unique())

for d in dates:
    df_window = cohort[
        cohort['cohort_date'] <= d
    ].copy()
    df_window['converted_cum'] = (
        (df_window['converted'] == 1) &
        (df_window['first_purchase_date'] <= d)
    ).astype(int)
    grouped = (
        df_window
        .groupby('group')['converted_cum']
        .agg(['sum', 'count'])
        .reset_index()
    )
    grouped.columns = ['group', 'conversions_cum', 'total_cum']
    grouped['cum_conversion'] = (grouped['conversions_cum'] / grouped['total_cum'].replace(0, pd.NA))
    grouped['date'] = d
    records.append(grouped)

df_cum = pd.concat(records, ignore_index = True)
df_cum['date'] = pd.to_datetime(df_cum['date'])
df_cum = df_cum.sort_values(['group', 'date'])

plt.figure(figsize = (12, 6))

for group in ['control', 'treatment']:
    if group in df_cum['group'].unique():
        df_plot = df_cum[df_cum['group'] == group]
        plt.plot(
            df_plot['date'],
            df_plot['cum_conversion'],
            marker = 'o',
            label = group,
            linewidth = 2
        )

plt.title('Конверсия в первую покупку пользователей TikTok', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('Конверсия в первую покупку')

plt.grid(True, linestyle = '--', alpha = 0.7)
plt.legend(title = 'Группа', fontsize = 11)
plt.xticks(rotation = 45)
plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))

plt.tight_layout()
plt.show()

Разница большая, p‑value показывает, что тест статистически значим. Сильный разрыв увеличивается со временем, разница между группами не снижается. Такая положительная разница может говорить об успехе предоставления бонуса на первую покупку новым пользователям, привлеченным через TikTok. Нулевую гипотезу отвергаем.

Теперь вспомогательная метрика ARPU. Комиссия платформы - 5%. Будем использовать односторонний t-test.

H0 - ARPU в тестовой группе не выше, чем в контрольной группе.

H1 - ARPU в тестовой группе выше, чем в контрольной группе.

In [ ]:
orders_21d['revenue'] = orders_21d['total_price'] * 0.05

user_revenue = (
    orders_21d
    .groupby('user_id')['revenue']
    .sum()
    .reset_index()
)

user_revenue.rename(columns = {'revenue': 'revenue_21d'}, inplace = True)

arpu_cohort = cohort[
    ['user_id', 'group']
].merge(
    user_revenue,
    on = 'user_id',
    how = 'left'
)

arpu_cohort['revenue_21d'] = (arpu_cohort['revenue_21d'].fillna(0))

arpu_agg = (
    arpu_cohort
    .groupby('group')['revenue_21d']
    .agg(['mean', 'count'])
    .rename(
        columns = {
            'mean': 'ARPU',
            'count': 'users'
        }
    )
)

arpu_control = arpu_cohort[arpu_cohort['group'] == 'control']['revenue_21d']
arpu_treatment = arpu_cohort[arpu_cohort['group'] == 'treatment']['revenue_21d']

t_stat, p_val = ttest_ind(
    arpu_treatment,
    arpu_control,
    equal_var = False,
    alternative = 'greater'
)

arpu_control_mean = arpu_agg.loc['control', 'ARPU']
arpu_treatment_mean = arpu_agg.loc['treatment', 'ARPU']
absolute_lift_arpu = (arpu_treatment_mean - arpu_control_mean)
relative_lift_arpu = (100 * (arpu_treatment_mean / arpu_control_mean - 1) if arpu_control_mean != 0 else 0)

results_arpu = pd.DataFrame([{
    'Metric': 'ARPU',
    'Group A': 'control',
    'Group B': 'treatment',
    'p-value': p_val,
    'ARPU A': arpu_control_mean,
    'ARPU B': arpu_treatment_mean,
    'Absolute Lift': absolute_lift_arpu,
    'Relative Lift (%)': relative_lift_arpu
}])

results_arpu

p‑value показывает, что разница между группами теста статистически значима.

In [ ]:
records = []

for d in dates:
    df_window = arpu_cohort[cohort['cohort_date'] <= d].copy()
    grouped = (
        df_window
        .groupby('group')['revenue_21d']
        .agg(['mean', 'count'])
        .reset_index()
    )
    grouped.columns = ['group', 'arpu', 'total_users']
    grouped['date'] = d
    records.append(grouped)

df_arpu_cum = pd.concat(records, ignore_index = True)
df_arpu_cum['date'] = pd.to_datetime(df_arpu_cum['date'])
df_arpu_cum = df_arpu_cum.sort_values(['group', 'date'])

plt.figure(figsize = (12, 6))

for group in ['control', 'treatment']:
    if group in df_arpu_cum['group'].unique():
        df_plot = df_arpu_cum[df_arpu_cum['group'] == group]
        plt.plot(
            df_plot['date'],
            df_plot['arpu'],
            marker = 'o',
            label = group,
            linewidth = 2
        )

plt.title('ARPU пользователей TikTok', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('ARPU')

plt.grid(True, linestyle = '--', alpha = 0.7)
plt.legend(title = 'Группа', fontsize = 11)
plt.xticks(rotation = 45)

plt.tight_layout()
plt.show()

Разница в ARPU есть, p‑value показывает, что тест статистически значим. Разрыв со временем немного уменьшается, линия устаканивается. Бонус на первую покупку увеличивает ARPU пользователей, привлеченных через TikTok. Нулевую гипотезу отвергаем.

Теперь вспомогательная метрика ARPPU.

H0 - ARPPU в тестовой группе не выше, чем в контрольной группе.

H1 - ARPPU в тестовой группе выше, чем в контрольной группе.

In [ ]:
user_revenue_arppu = (
    orders_21d
    .groupby('user_id')['revenue']
    .sum()
    .reset_index()
)

user_revenue_arppu.rename(columns = {'revenue': 'revenue_21d'},inplace = True)

user_revenue_arppu = user_revenue_arppu.merge(
    df_split_ab[['user_id', 'group']],
    on = 'user_id',
    how = 'left',
    validate = 'many_to_one'
)

arppu_agg = (
    user_revenue_arppu
    .groupby('group')['revenue_21d']
    .agg(['mean', 'count'])
    .rename(
        columns = {
            'mean': 'ARPPU',
            'count': 'buyers'
        }
    )
)

arppu_control = user_revenue_arppu[user_revenue_arppu['group'] == 'control']['revenue_21d']

arppu_treatment = user_revenue_arppu[user_revenue_arppu['group'] == 'treatment']['revenue_21d']

t_stat, p_val = ttest_ind(
    arppu_treatment,
    arppu_control,
    equal_var = False,
    alternative = 'greater'
)

arppu_control_mean = arppu_agg.loc['control', 'ARPPU']
arppu_treatment_mean = arppu_agg.loc['treatment', 'ARPPU']
absolute_lift_arppu = (arppu_treatment_mean - arppu_control_mean)
relative_lift_arppu = (100 * (arppu_treatment_mean / arppu_control_mean - 1) if arppu_control_mean != 0 else 0)

results_arppu = pd.DataFrame([{
    'Metric': 'ARPPU',
    'Group A': 'control',
    'Group B': 'treatment',
    'p-value': p_val,
    'ARPPU A': arppu_control_mean,
    'ARPPU B': arppu_treatment_mean,
    'Absolute Lift': absolute_lift_arppu,
    'Relative Lift (%)': relative_lift_arppu
}])

results_arppu

p‑value показывает, что разница между группами теста статистически не значима, но на самой грани.

In [ ]:
records = []

arppu_data = user_revenue_arppu.copy()

arppu_data = arppu_data.merge(
    df_abt[['user_id', 'registration_date']],
    on = 'user_id',
    how = 'left',
    validate = 'one_to_one'
)

arppu_data['cohort_date'] = (arppu_data['registration_date'].dt.date)
dates = sorted(arppu_data['cohort_date'].unique())

for d in dates:
    df_window = arppu_data[arppu_data['cohort_date'] <= d].copy()
    grouped = (
        df_window
        .groupby('group')['revenue_21d']
        .agg(['mean', 'count'])
        .reset_index()
    )
    grouped.columns = ['group', 'arppu', 'buyers']
    grouped['date'] = d
    records.append(grouped)

df_arppu_cum = pd.concat(records, ignore_index = True)
df_arppu_cum['date'] = pd.to_datetime(df_arppu_cum['date'])
df_arppu_cum = df_arppu_cum.sort_values(['group', 'date'])

plt.figure(figsize = (12, 6))

for group in ['control', 'treatment']:
    if group in df_arppu_cum['group'].unique():
        df_plot = df_arppu_cum[df_arppu_cum['group'] == group]
        plt.plot(
            df_plot['date'],
            df_plot['arppu'],
            marker = 'o',
            label = group,
            linewidth = 2
        )

plt.title('ARPPU пользователей TikTok', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('ARPPU')

plt.grid(True, linestyle = '--', alpha = 0.7)
plt.legend(title = 'Группа', fontsize = 11)
plt.xticks(rotation = 45)

plt.tight_layout()
plt.show()

Нулевую гипотезу нельзя отвергнуть, хоть и ARPPU в тестовой группе выше. К концу графика значения выравниваются.

Рассчитаем вспомогательную метрику AOV.

H0 - AOV в тестовой группе не выше, чем в контрольной группе.

H1 - AOV в тестовой группе выше, чем в контрольной группе.

In [ ]:
user_aov = (
    orders_21d
    .groupby(['user_id', 'group'])['total_price']
    .mean()
    .reset_index()
    .rename(columns = {'total_price': 'aov_21d'})
)

aov_agg = (
    user_aov
    .groupby('group')['aov_21d']
    .agg(['mean', 'count'])
    .rename(
        columns = {
            'mean': 'AOV',
            'count': 'users'
        }
    )
)

aov_control = user_aov[user_aov['group'] == 'control']['aov_21d']
aov_treatment = user_aov[user_aov['group'] == 'treatment']['aov_21d']

t_stat, p_val = ttest_ind(
    aov_treatment,
    aov_control,
    equal_var = False,
    alternative = 'greater'
)

aov_control_mean = aov_agg.loc['control', 'AOV']
aov_treatment_mean = aov_agg.loc['treatment', 'AOV']
absolute_lift_aov = (aov_treatment_mean - aov_control_mean)
relative_lift_aov = (100 * (aov_treatment_mean / aov_control_mean - 1) if aov_control_mean != 0 else 0)

results_aov = pd.DataFrame([{
    'Metric': 'AOV',
    'Group A': 'control',
    'Group B': 'treatment',
    'p-value': p_val,
    'AOV A': aov_control_mean,
    'AOV B': aov_treatment_mean,
    'Absolute Lift': absolute_lift_aov,
    'Relative Lift (%)': relative_lift_aov
}])

results_aov

p‑value показывает, что разница между группами теста статистически не значима.

In [ ]:
records = []

aov_data = user_aov.merge(
    df_abt[['user_id', 'registration_date']],
    on = 'user_id',
    how = 'left',
    validate = 'one_to_one'
)

aov_data['cohort_date'] = (aov_data['registration_date'].dt.date)
dates = sorted(aov_data['cohort_date'].unique())

for d in dates:
    df_window = aov_data[aov_data['cohort_date'] == d].copy()
    grouped = (
        df_window
        .groupby('group')['aov_21d']
        .agg(['mean', 'count'])
        .reset_index()
    )
    grouped.columns = ['group', 'aov', 'users']
    grouped['date'] = d
    records.append(grouped)

df_aov_cum = pd.concat(records, ignore_index = True)
df_aov_cum['date'] = pd.to_datetime(df_aov_cum['date'])
df_aov_cum = df_aov_cum.sort_values(['group', 'date'])

plt.figure(figsize = (12, 6))

for group in ['control', 'treatment']:
    if group in df_aov_cum['group'].unique():
        df_plot = df_aov_cum[df_aov_cum['group'] == group]
        plt.plot(
            df_plot['date'],
            df_plot['aov'],
            marker = 'o',
            label = group,
            linewidth = 2
        )

plt.title('AOV пользователей TikTok', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('AOV')

plt.grid(True, linestyle = '--', alpha = 0.7)
plt.legend(title = 'Группа', fontsize = 11)
plt.xticks(rotation = 45)

plt.tight_layout()
plt.show()

Нулевую гипотезу нельзя отвергнуть, снижение среднего чека из-за бонуса статистически не значимо.

Теперь изучим барьерную мертику.

H0 - Средняя выручка на пользователя тестовой группе не выросла.

H1 - Средняя выручка на пользователя тестовой группе выросла.

In [ ]:
revenue_agg = (
    orders_21d
    .groupby('group')['revenue']
    .agg(['sum', 'count'])
    .rename(
        columns = {
            'sum': 'total_revenue',
            'count': 'orders'
        }
    )
)

user_revenue = (
    orders_21d
    .groupby('user_id')['revenue']
    .sum()
    .reset_index()
    .rename(columns = {'revenue': 'revenue_21d'})
)

user_revenue = user_revenue.merge(
    df_split_ab[['user_id', 'group']],
    on = 'user_id',
    how = 'right',
    validate = 'one_to_one'
)

user_revenue['revenue_21d'] = (user_revenue['revenue_21d'].fillna(0))

revenue_user_agg = (
    user_revenue
    .groupby('group')['revenue_21d']
    .agg(['mean', 'count'])
    .rename(
        columns = {
            'mean': 'revenue_per_user',
            'count': 'users'
        }
    )
)

revenue_control = user_revenue[user_revenue['group'] == 'control']['revenue_21d']
revenue_treatment = user_revenue[user_revenue['group'] == 'treatment']['revenue_21d']

t_stat, p_val = ttest_ind(
    revenue_treatment,
    revenue_control,
    equal_var = False,
    alternative = 'greater'
)

revenue_per_user_control = revenue_user_agg.loc['control', 'revenue_per_user']
revenue_per_user_treatment = revenue_user_agg.loc['treatment', 'revenue_per_user']
absolute_lift_revenue = (revenue_per_user_treatment - revenue_per_user_control)
relative_lift_revenue = (100 * (revenue_per_user_treatment / revenue_per_user_control - 1) if revenue_per_user_control != 0 else 0)

results_revenue = pd.DataFrame([{
    'Metric': 'Revenue per User',
    'Group A': 'control',
    'Group B': 'treatment',
    'p-value': p_val,
    'Revenue per User A': revenue_per_user_control,
    'Revenue per User B': revenue_per_user_treatment,
    'Absolute Lift': absolute_lift_revenue,
    'Relative Lift (%)': relative_lift_revenue
}])

results_revenue

p‑value показывает, что разница между группами теста статистически значима.

In [ ]:
records = []

revenue_data = user_revenue.merge(
    df_abt[['user_id', 'registration_date']],
    on = 'user_id',
    how = 'left',
    validate = 'one_to_one'
)

revenue_data['cohort_date'] = (revenue_data['registration_date'].dt.date)
dates = sorted(revenue_data['cohort_date'].unique())

for d in dates:
    df_day = revenue_data[revenue_data['cohort_date'] == d].copy()
    grouped = (
        df_day
        .groupby('group')['revenue_21d']
        .agg(['mean', 'count'])
        .reset_index()
    )
    grouped.columns = ['group', 'revenue_per_user', 'users']
    grouped['date'] = d
    records.append(grouped)

df_revenue_daily = pd.concat(records, ignore_index = True)
df_revenue_daily['date'] = pd.to_datetime(df_revenue_daily['date'])
df_revenue_daily = df_revenue_daily.sort_values(['group', 'date'])

plt.figure(figsize = (12, 6))

for group in ['control', 'treatment']:
    df_plot = df_revenue_daily[df_revenue_daily['group'] == group]
    plt.plot(
        df_plot['date'],
        df_plot['revenue_per_user'],
        marker = 'o',
        label = group,
        linewidth = 2
    )

plt.title('Средняя выручка на пользователя', fontsize = 14)
plt.xlabel('Даты')
plt.ylabel('Выручка')

plt.grid(True, linestyle = '--', alpha = 0.7)
plt.legend(title = 'Группа', fontsize = 11)
plt.xticks(rotation = 45)

plt.tight_layout()
plt.show()

Барьерная метрика не ухудшидась, можно отвергнуть нулевую гипотезу.

### Анализ изменения трат

Используя бутстрап, сравним разницы перцентилей (25, 50, 75) суммы всех покупок платящих пользователей в контрольной и тестовой группах.

In [ ]:
user_orders_sum = (
    orders_21d
    .groupby('user_id')['total_price']
    .sum()
    .reset_index()
    .rename(columns = {'total_price': 'orders_sum_21d'})
)

user_orders_sum = user_orders_sum.merge(
    df_split_ab[['user_id', 'group']],
    on = 'user_id',
    how = 'right',
    validate = 'one_to_one'
)

user_orders_sum['orders_sum_21d'] = (user_orders_sum['orders_sum_21d'].fillna(0))
control = user_orders_sum[(user_orders_sum['group'] == 'control') & (user_orders_sum['orders_sum_21d'] > 0)]['orders_sum_21d'].values
treatment = user_orders_sum[(user_orders_sum['group'] == 'treatment') & (user_orders_sum['orders_sum_21d'] > 0)]['orders_sum_21d'].values

In [ ]:
np.random.seed(41)

n_iterations = 1000
boot_diffs = []

for i in range(n_iterations):
    boot_control = np.random.choice(
        control,
        size = len(control),
        replace = True
    )
    boot_treatment = np.random.choice(
        treatment,
        size = len(treatment),
        replace = True
    )
    control_25p = np.percentile(boot_control, 25)
    treatment_25p = np.percentile(boot_treatment, 25)
    boot_diffs.append(treatment_25p - control_25p)

boot_diffs = np.array(boot_diffs)
diff_ci = np.percentile(boot_diffs, [2.5, 97.5])

print(f"25-й перцентиль контрольной группы: "f"{np.percentile(control, 25):.2f}")
print(f"25-й перцентиль тестовой группы: "f"{np.percentile(treatment, 25):.2f}")
print(f"95%-й доверительный интервал разности 25-х перцентилей "f"[{diff_ci[0]:.2f}, {diff_ci[1]:.2f}]")

In [ ]:
plt.figure(figsize = (12, 6))

plt.hist(boot_diffs, bins = 30, color = 'salmon', edgecolor = 'black')
plt.axvline(diff_ci[0], color = 'red', linestyle = '--', label = '2.5-й перцентиль')
plt.axvline(diff_ci[1], color = 'red', linestyle = '--', label = '97.5-й перцентиль')
plt.title("Бутстрап-распределение 25-х перцентилей")
plt.legend()
plt.show()

Весь доверительный интервал находится ниже 0, значит различие статистически значимо. Получается бонус конвертирурет в покупку больше пользователей, среди них больше людей с небольшими суммами заказов.

In [ ]:
boot_diffs = []

for i in range(n_iterations):
    boot_control = np.random.choice(
        control,
        size = len(control),
        replace = True
    )
    boot_treatment = np.random.choice(
        treatment,
        size = len(treatment),
        replace = True
    )
    control_50p = np.percentile(boot_control, 50)
    treatment_50p = np.percentile(boot_treatment, 50)
    boot_diffs.append(treatment_50p - control_50p)

boot_diffs = np.array(boot_diffs)
diff_ci = np.percentile(boot_diffs, [2.5, 97.5])

print(f"50-й перцентиль контрольной группы: "f"{np.percentile(control, 50):.2f}")
print(f"50-й перцентиль тестовой группы: "f"{np.percentile(treatment, 50):.2f}")
print(f"95%-й доверительный интервал разности 50-х перцентилей "f"[{diff_ci[0]:.2f}, {diff_ci[1]:.2f}]")

In [ ]:
plt.figure(figsize = (12, 6))

plt.hist(boot_diffs, bins = 30, color = 'salmon', edgecolor = 'black')
plt.axvline(diff_ci[0], color = 'red', linestyle = '--', label = '2.5-й перцентиль')
plt.axvline(diff_ci[1], color = 'red', linestyle = '--', label = '97.5-й перцентиль')
plt.title("Бутстрап-распределение 50-х перцентилей")
plt.legend()
plt.show()

Интервал содержит 0, так что значимых изменений нет. Скорее пурукос влево из-за небольших сумм.

In [ ]:
boot_diffs = []

for i in range(n_iterations):
    boot_control = np.random.choice(
        control,
        size = len(control),
        replace = True
    )
    boot_treatment = np.random.choice(
        treatment,
        size = len(treatment),
        replace = True
    )
    control_75p = np.percentile(boot_control, 75)
    treatment_75p = np.percentile(boot_treatment, 75)
    boot_diffs.append(treatment_75p - control_75p)

boot_diffs = np.array(boot_diffs)
diff_ci = np.percentile(boot_diffs, [2.5, 97.5])

print(f"75-й перцентиль контрольной группы: "f"{np.percentile(control, 75):.2f}")
print(f"75-й перцентиль тестовой группы: "f"{np.percentile(treatment, 75):.2f}")
print(f"95%-й доверительный интервал разности 75-х перцентилей "f"[{diff_ci[0]:.2f}, {diff_ci[1]:.2f}]")

In [ ]:
plt.figure(figsize = (12, 6))

plt.hist(boot_diffs, bins = 30, color = 'salmon', edgecolor = 'black')
plt.axvline(diff_ci[0], color = 'red', linestyle = '--', label = '2.5-й перцентиль')
plt.axvline(diff_ci[1], color = 'red', linestyle = '--', label = '97.5-й перцентиль')
plt.title("Бутстрап-распределение 75-х перцентилей")
plt.legend()
plt.show()

Тоже статистически не значимое изменение. Бонус, вероятно, увеличивает конверсию за счёт пользователей с меньшим объёмом покупок.

Воронка тестовой группы лучше, больше пользователей переходят к чекауту и совершают покупку. Основная метрика кратно выросла - предоставление бонуса на первую покупку новым пользователям, привлеченным через TikTok, приводит к увеличению конверсии в первую покупку, этот эффект статистически значим. Вспомогательные метрики показали, что бонус побуждает к самой первой покупке, но статистически значимо не увеличивает средний доход с покупателя, не увеличивает средний чек. Но бонус и не ухудшает монетизацию, вредных эффектов нет. Барьерная метрика не ухудшидась.

## Итоги исследования

В целом эксперимент можно считать успешным.

Гипотеза о предоставлении бонуса на первую покупку новым пользователям, привлеченным через TikTok, судя по данным, приводит к увеличению конверсии в первую покупку. Среди них появляется больше покупателей с небольшими суммами заказов. Статистически значимого ухудшения монетизации нет.

Рекомендуется проверить, сколько внедрение бонуса будет стоить, на сколько вырастет маркетинговый бюджет (на ТикТок и так выделяется очень много денег), окупится ли. Если всё в порядке, масштабиорвать бонус на весь канал.